In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import time
import warnings
import soccerdata as sd
from datetime import datetime
import re
import difflib
import unicodedata

In [ ]:
data_merged_gw_2016_17 = pd.read_csv('data/2016-17/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2017_18 = pd.read_csv('data/2017-18/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2018_19 = pd.read_csv('data/2018-19/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2019_20 = pd.read_csv('data/2019-20/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2020_21 = pd.read_csv('data/2020-21/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2021_22 = pd.read_csv('data/2021-22/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2022_23 = pd.read_csv('data/2022-23/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2023_24 = pd.read_csv('data/2023-24/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2024_25 = pd.read_csv('data/2024-25/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')
data_merged_gw_2025_26 = pd.read_csv('data/2025-26/gws/merged_gw.csv', encoding='latin-1', on_bad_lines='skip')

## adding position column 

### load the players tables for the target seasons

## add the position column

## adding team name to the dataset

In [ ]:
# apply the function to each season's merged gw data
add_team_name_to_merged_gw(data_merged_gw_2016_17, data_players_2016_17, data_master_team_list, '2016-17')
add_team_name_to_merged_gw(data_merged_gw_2017_18, data_players_2017_18, data_master_team_list, '2017-18')
add_team_name_to_merged_gw(data_merged_gw_2018_19, data_players_2018_19, data_master_team_list, '2018-19')
add_team_name_to_merged_gw(data_merged_gw_2019_20, data_players_2019_20, data_master_team_list, '2019-20')

In [ ]:
# Drop the xP (expected points) column from seasons 2020-21 through 2025-26 for consistency
data_merged_gw_2020_21 = data_merged_gw_2020_21.drop(columns=['xP'])
data_merged_gw_2021_22 = data_merged_gw_2021_22.drop(columns=['xP'])
data_merged_gw_2022_23 = data_merged_gw_2022_23.drop(columns=['xP'])
data_merged_gw_2023_24 = data_merged_gw_2023_24.drop(columns=['xP'])
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['xP'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['xP'])
# Drop columns not available across all seasons (2016-17 to 2018-19)
data_merged_gw_2016_17 = data_merged_gw_2016_17.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
data_merged_gw_2017_18 = data_merged_gw_2017_18.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
data_merged_gw_2018_19 = data_merged_gw_2018_19.drop(columns=['attempted_passes','big_chances_created','big_chances_missed','completed_passes','dribbles','ea_index','errors_leading_to_goal','errors_leading_to_goal_attempt','fouls','key_passes','kickoff_time_formatted','loaned_in','loaned_out','offside','open_play_crosses','penalties_conceded','tackled','target_missed','winning_goals'])
# Drop from 2022 the xp and that stuff
data_merged_gw_2022_23 = data_merged_gw_2022_23.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2023_24 = data_merged_gw_2023_24.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['expected_goals','expected_assists','expected_goals_conceded','expected_goal_involvements','starts'])
# Drop modified in the last two seasons
data_merged_gw_2024_25 = data_merged_gw_2024_25.drop(columns=['modified'])
data_merged_gw_2025_26 = data_merged_gw_2025_26.drop(columns=['modified'])
# drop the id from the first two seasons
data_merged_gw_2016_17 = data_merged_gw_2016_17.drop(columns=['id'])
data_merged_gw_2017_18 = data_merged_gw_2017_18.drop(columns=['id'])
data_merged_gw_2018_19 = data_merged_gw_2018_19.drop(columns=['id'])

In [ ]:
# Calculate defensive_contribution for seasons 2016-17 to 2024-25
# FPL scoring rules:
# - Defenders: clearances_blocks_interceptions + tackles
# - Midfielders/Forwards: clearances_blocks_interceptions + tackles + recoveries
def add_defensive_contribution(merged_gw_df):
    def calculate_defensive_contribution(row):
        position = row['position']
        clearances = row.get('clearances_blocks_interceptions', 0)
        tackles = row.get('tackles', 0)
        recoveries = row.get('recoveries', 0)
        if position == 'Defender':
            return clearances + tackles
        elif position in ['Midfielder', 'Forward']:
            return clearances + tackles + recoveries
        else:
            return 0
    merged_gw_df['defensive_contribution'] = merged_gw_df.apply(calculate_defensive_contribution, axis=1)
    return merged_gw_df

### verify that all the cols are identical now

# hsitorical points adjustment

### consider moving this to the end (after merging with the defensive data)

In [ ]:
# Adjust historical points (2016-17 to 2018-19) to align with current FPL scoring system
# Add 2 bonus points when:
# - Defenders reach defensive_contribution >= 10
# - Midfielders/Forwards reach defensive_contribution >= 12
def modify_points(merged_gw_df):
    def calculate_modified_points(row):
        points = row['total_points']
        position = row['position']
        defensive_contribution = row['defensive_contribution']
        if position == 'Defender' and defensive_contribution >= 10:
            points += 2
        elif position in ['Midfielder', 'Forward'] and defensive_contribution >= 12:
            points += 2
        return points
    merged_gw_df['total_points'] = merged_gw_df.apply(calculate_modified_points, axis=1)
    return merged_gw_df
data_merged_gw_2016_17 = modify_points(data_merged_gw_2016_17)
data_merged_gw_2017_18 = modify_points(data_merged_gw_2017_18)
data_merged_gw_2018_19 = modify_points(data_merged_gw_2018_19)

# merging all seasons data in a single dataset

## standarizing the position across seasons

## converting the opponent team from id to name

# saving the current state of data as csv

# here we add the game number feature then we make the merging with defensive stats

## Load Datasets

## Step 1: Build Team-Game Mapping from fixtures.csv

For each season (2018-19 to 2024-25):
1. Load fixtures.csv and teams.csv
2. Create team_id → team_name mapping
3. Convert each fixture to 2 team-game records (home + away)
4. Sort chronologically and assign game_number 1-38

## Step 2: Join game_number to all_seasons_data.csv

Join using the fixture ID (100% match rate for seasons with fixtures.csv)

## Step 3: Add game_number to defensive_stats_raw.csv

Since defensive_stats doesn't have fixture IDs, we'll use date-based matching:
1. Parse the 'game' column to extract date and teams
2. Match with fixtures using date + team combination

## Step 5: Save Updated Datasets

# now merging the main table with the defensive stats table

In [ ]:
# Load the raw defensive data
print("Loading defensive stats raw data...")
df_def = pd.read_csv('defensive_stats_raw.csv', low_memory=False)
print(f"Original defensive data shape: {df_def.shape}")

# Remove the header row (row 0 contains column descriptions)
df_def = df_def[df_def['season'].notna() & (df_def['season'] != '')]
df_def = df_def.reset_index(drop=True)
print(f"After removing header row: {df_def.shape}")

In [ ]:
# 2. COMBINE TACKLE COLUMNS (Def 3rd, Mid 3rd, Att 3rd -> Total Tackles)
print("="*60)
print("COMBINING TACKLE COLUMNS")
print("="*60)

# The columns are named 'Tackles.2' (Def 3rd), 'Tackles.3' (Mid 3rd), 'Tackles.4' (Att 3rd)
# Convert to numeric and combine
tackle_cols = ['Tackles.2', 'Tackles.3', 'Tackles.4']
for col in tackle_cols:
    df_def[col] = pd.to_numeric(df_def[col], errors='coerce')

# Create combined tackles column (sum of all three thirds)
df_def['tackles_total'] = df_def[tackle_cols].sum(axis=1)

print(f"✓ Created 'tackles_total' by combining tackles from all thirds")
print(f"  Sample values: {df_def['tackles_total'].head(10).tolist()}")

In [ ]:
# 3. SELECT AND RENAME DEFENSIVE COLUMNS
print("="*60)
print("SELECTING DEFENSIVE COLUMNS")
print("="*60)

# Map raw columns to FPL-style naming
column_mapping = {
    'season': 'season',
    'player': 'name',
    'team': 'team',
    'pos': 'position',
    'min': 'minutes',
    'Tackles': 'tackles',  # Total tackles (Tkl)
    'Tackles.1': 'tackles_won',  # TklW
    'tackles_total': 'tackles_total',  # Combined Def+Mid+Att third
    'Challenges': 'challenges',  # Total challenges
    'Challenges.1': 'challenges_attempted',  # Att
    'Challenges.2': 'challenges_success_rate',  # Tkl%
    'Challenges.3': 'challenges_lost',  # Lost
    'Blocks': 'blocks',  # Total blocks
    'Blocks.1': 'blocks_shots',  # Sh
    'Blocks.2': 'blocks_passes',  # Pass
    'Int': 'interceptions',  # Interceptions
    'Tkl+Int': 'tackles_interceptions',  # Tkl+Int
    'Clr': 'clearances',  # Clearances
    'Err': 'errors',  # Errors leading to shots
    'match_id': 'match_id',
    'game': 'game'
}

# Select only the columns we need for defensive stats
defensive_columns = list(column_mapping.keys())
df_def_selected = df_def[defensive_columns].copy()

# Rename columns to match FPL naming
df_def_selected = df_def_selected.rename(columns=column_mapping)

print(f"Selected columns: {df_def_selected.columns.tolist()}")

# Assign gameweek

## save the cleaned data into csv

# Now the merging part:

In [ ]:
# loading both datasets
df_def = pd.read_csv("defensive_stats_cleaned.csv")
df_main = pd.read_csv("all_seasons_data.csv")

In [ ]:

# ==========================================
# STEP 1: FORCE-CLEAN NAMES
# ==========================================

def clean_main_name_format(name):
    if not isinstance(name, str):
        return str(name)
    
    # 1. Fix Mojibake (Encoding Errors)
    char_map = {
        'Ã©': 'é', 'Ãº': 'ú', 'Ã¡': 'á', 'Ã³': 'ó', 'Ã¨': 'è', 'Ã±': 'ñ',
        'Ã\xad': 'í', 'Ã§': 'ç', 'Ã¢': 'â', 'Ã¼': 'ü', 'Ã¶': 'ö', 'Ã\x9f': 'ß',
        'Ã¸': 'ø', 'Ã«': 'ë', 'Ã': 'à' ,'à£': 'ã', 'à©': 'é'
    }
    for bad, good in char_map.items():
        name = name.replace(bad, good)

    # 2. Remove trailing IDs (e.g., '_376', '_12', '_4')
    # Regex: Underscore followed by 1 or more digits at the END of the string
    name = re.sub(r'_\d+$', '', name)
    
    # 3. Replace remaining underscores with spaces
    name = name.replace('_', ' ')
    
    # 4. Standardize (lower, strip)
    return name.lower().strip()

# Create/Overwrite the 'join_name' column
df_main['join_name'] = df_main['name'].apply(clean_main_name_format)


In [ ]:
# first filter only the seasons that we need from defensive data
seasons_needed = df_def['season'].unique()
df_main = df_main[df_main['season'].isin(seasons_needed)]
# print the target seasons
print("Seasons needed for merging:", seasons_needed)

In [ ]:
#clearances_blocks_interceptions, recoveries, defensive_contribution, tackles are the cols needed to be added to the defensive df

def add_clearances_blocks_interceptions(df):
    df['clearances_blocks_interceptions'] = df['clearances'] + df['blocks'] + df['interceptions']
    return df
df_def = add_clearances_blocks_interceptions(df_def)
# print sample data to verify
print(df_def[['clearances', 'blocks', 'interceptions', 'clearances_blocks_interceptions']].head())

In [ ]:
df_def['join_name'] = df_def['name'].astype(str).str.lower().str.strip()

# 2. Standardize Seasons (Ensure they match "2023-24" format in both)
df_main['join_season'] = df_main['season'].astype(str).str.strip()
df_def['join_season'] = df_def['season'].astype(str).str.strip()

manual_nickname_map = {
    'jorge luiz frello filho': 'jorginho',
    'jonathan castro otto': 'jonny',
    'bruno guimaraes rodriguez moura': 'bruno guimaraes',
    'gabriel teodoro martinelli silva': 'gabriel martinelli',
    'emerson leie de souza junior': 'emerson royal',
    'david raya martin': 'david raya',
    'jose sa': 'jose sa',
    'joao palhinha goncalves alves': 'joao palhinha',
    'thiago alcantara do nascimento': 'thiago',
    'matheus luiz nunes': 'matheus nunes',
    'antony matheus dos santos': 'antony',
    'richarlison de andrade': 'richarlison',
    'bernardo mota veiga de carvalho e silva': 'bernardo silva',
    'ederson santana de moraes': 'ederson',
    'joão filipe iria santos moutinho': 'joão moutinho',
    'fabio henrique tavares': 'fabinho',
    'frederico rodrigues de paula santos': 'fred',
    'lucas tolentino coelho de lima': 'lucas paquetá', # Check accent
    'lucas tolentino coelho de lima': 'lucas paqueta', # Try both if unsure
    'benjamin white': 'ben white',
    'gabriel dos santos magalhães': 'gabriel magalhães', # Now that encoding is fixed, map to short
    'bruno guimarães rodriguez moura': 'bruno guimarães',
    'joão palhinha gonçalves': 'joão palhinha',
    'tomas soucek': 'tomáš souček', # Add accents if defensive has them
    'casemiro': 'casemiro', # If Main has long name, map it. Likely 'carlos henrique casimiro'
    'carlos henrique casimiro': 'casemiro',
    'norberto bercique gomes betuncal': 'beto',
    'jose angel esmoris tasende': 'angelino',
    'juan camilo hernandez suarez': 'cucho',
    'daniel ceballos fernandez': 'dani ceballos',
    'anssumane fati vieira': 'ansu fati',
    'anssumane fati': 'ansu fati',
    'alexandre moreno lopera': 'alex moreno',
    'łukasz fabianski': 'lukasz fabianski', # Fix the Polish 'ł' manually
    'lukasz fabianski': 'lukasz fabianski',  # Safety net
    'ahmed el-sayed hegazy': 'ahmed hegazi',
    'a\x81lex moreno lopera': 'alex moreno',   # Found in your list
    'ivan peria¡ia\x87': 'ivan perisic',       # Found in your list
    'muhamed bea¡ia\x87': 'muhamed besic',     # Found in your list
    'a\x81lex moreno lopera': 'alex moreno',
    'edson a\x81lvarez velazquez': 'edson alvarez',
    
    # The Nicknames & Legal Names
    'abdul fatawu': 'abdul fatawu issahaku',
    'borja gonzalez tomas': 'borja baston',
    'fabio ferreira vieira': 'fabio vieira',
    'fernando luiz rosa': 'fernandinho',
    'giovanni reyna': 'gio reyna',
    'hamed traore': 'hamed junior traore',
    'ian carlo poveda-ocampo': 'ian poveda',
    'jhon duran': 'jader duran',
    'julian araujo zuniga': 'julian araujo',
    'thakgalo leshabela': 'khanya leshabela',
    'francisco casilla cortes': 'kiko casilla',
    'francisco femenia far': 'kiko femenia',
    'marcus oliveira alencar': 'marquinhos',
    'oluwasemilogo adesewo ibidapo ajayi': 'semi ajayi',
    'tariqe fosu-henry': 'tariqe fosu',
    'vini de souza costa': 'vinicius souza',
    'vitor ferreira': 'vitinha',
    'jose reina': 'pepe reina',
    'djordje petrovic': 'đorđe petrovic',  # Matching the defensive spelling
    'jose a\x81ngel esmoris tasende': 'angelino', # Found hidden in candidate list
}

df_main['join_name'] = df_main['join_name'].replace(manual_nickname_map)


# ==========================================
# STEP 3: AUTOMATED SMART MATCHING
# ==========================================
print("--- STARTING SMART MATCHING ---")

# 1. PREPARE LISTS
# We only care about names that are currently missing in the Main DF
# (i.e., names in Main that don't yet match a name in Defensive)
valid_def_names = set(df_def['join_name'].unique())
main_unique = df_main['join_name'].unique()
missing_names = [n for n in main_unique if n not in valid_def_names]

print(f"Attempting to resolve {len(missing_names)} missing names...")

name_mapping = {}

# 2. LOGIC A: SUBSTRING MATCH (The "Gabriel Jesus" Fix)
# We check if a Defensive Name (Short) is fully inside a Main Name (Long)
# e.g. "gabriel jesus" is inside "gabriel fernando de jesus"
for m_name in missing_names:
    m_tokens = set(m_name.split())
    candidates = []
    
    for d_name in valid_def_names:
        d_tokens = set(d_name.split())
        # Check if ALL words in the short name appear in the long name
        if d_tokens.issubset(m_tokens):
            candidates.append(d_name)
    
    if candidates:
        # If multiple matches, pick the longest one (Most specific)
        # Prevents "Gabriel" matching "Gabriel Jesus" incorrectly
        best_match = max(candidates, key=len)
        name_mapping[m_name] = best_match

# 3. LOGIC B: FUZZY MATCH (The "Fabian Schär" Fix)
# For names that didn't match via substring (likely due to spelling/encoding diffs)
# We only check names that Logic A didn't solve
remaining_missing = [n for n in missing_names if n not in name_mapping]
def_name_list = list(valid_def_names)

for m_name in remaining_missing:
    # Cutoff 0.8 is strict to avoid bad matches (we prefer missing data over wrong data)
    matches = difflib.get_close_matches(m_name, def_name_list, n=1, cutoff=0.8)
    if matches:
        name_mapping[m_name] = matches[0]

# 4. APPLY THE UPDATES (To the JOIN KEY only)
print(f"Found {len(name_mapping)} new automatic matches.")
print("Updating 'join_name' column (Original names are safe)...")
df_main['join_name'] = df_main['join_name'].replace(name_mapping)


# ==========================================
# 1. DEFINE ACCENT REMOVER
# ==========================================
def remove_accents(input_str):
    if not isinstance(input_str, str):
        return str(input_str)
    # Normalize unicode characters to decompose them (e.g., 'á' becomes 'a' + '´')
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    # Filter out non-spacing mark characters (the accents)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

print("--- STRIPPING ACCENTS FROM BOTH DATASETS ---")

# Apply to Main
df_main['join_name'] = df_main['join_name'].apply(remove_accents)

# Apply to Defensive
df_def['join_name'] = df_def['join_name'].apply(remove_accents)



In [ ]:
print("\n=== UNIQUE NAME COVERAGE REPORT ===")

# Get unique names
main_names_set = set(df_main['join_name'].unique())
def_names_set = set(df_def['join_name'].unique())

# Calculate intersection
matched_names = main_names_set.intersection(def_names_set)
missing_def_names = def_names_set - main_names_set

# Metrics
total_def_names = len(def_names_set)
matched_count = len(matched_names)
coverage_pct = (matched_count / total_def_names) * 100

print(f"Unique Defensive Names:   {total_def_names}")
print(f"Found in Main DataFrame:  {matched_count}")
print(f"Defensive Name Coverage:  {coverage_pct:.2f}%")

In [ ]:
# ==========================================
# 2. THE MERGE & SMART VALIDATION
# ==========================================
# IMPORTANT: We now use game_number instead of GW for merging
# This handles postponed matches correctly by matching on chronological game order
print("--- MERGE DIAGNOSTICS (Using game_number) ---")

# 1. PREPARE DEFENSIVE DATA
cols_cbi = ['clearances', 'blocks', 'interceptions']
df_def[cols_cbi] = df_def[cols_cbi].fillna(0)

if 'clearances_blocks_interceptions' not in df_def.columns:
    df_def['clearances_blocks_interceptions'] = (
        df_def['clearances'] + df_def['blocks'] + df_def['interceptions']
    )

# Select Merge Subset - NOW USING game_number INSTEAD OF GW
def_subset = df_def[[
    'join_name', 'game_number', 'join_season', 
    'tackles', 'clearances_blocks_interceptions'
]].rename(columns={
    'tackles': 'tackles_new', 
    'clearances_blocks_interceptions': 'cbi_new'
})

# ---------------------------------------------------------
# SMART METRIC: "Can we match it?"
# ---------------------------------------------------------
# Using game_number for matching ensures correct alignment even with postponed matches
main_keys = set(zip(df_main['join_name'], df_main['game_number'], df_main['join_season']))
def_keys = set(zip(def_subset['join_name'], def_subset['game_number'], def_subset['join_season']))

# The Intersection: These are the rows that SHOULD merge successfully
possible_matches = main_keys.intersection(def_keys)
print(f"Total Rows in Main: {len(df_main)}")
print(f"Rows with available Defensive Data: {len(possible_matches)}")

# 2. PERFORM LEFT MERGE - NOW ON game_number
merged_df = pd.merge(
    df_main, 
    def_subset, 
    on=['join_name', 'game_number', 'join_season'], 
    how='left'
)

# ---------------------------------------------------------
# REAL VALIDATION: DID THE MERGE WORK?
# ---------------------------------------------------------
merged_df['key_tuple'] = list(zip(merged_df['join_name'], merged_df['game_number'], merged_df['join_season']))
should_have_data = merged_df[merged_df['key_tuple'].isin(possible_matches)]

# Check if they are actually filled
successful_merges = should_have_data['tackles_new'].notna().sum()
technical_success_rate = (successful_merges / len(should_have_data)) * 100 if len(should_have_data) > 0 else 0

print(f"\nTechnical Merge Success Rate: {technical_success_rate:.2f}%")
print("(This should be 100%. It means every row that existed in the source was successfully merged.)")

# ==========================================
# 3. UPDATE STATS & FINAL REPORT
# ==========================================
# Update Tackles
merged_df['tackles'] = np.where(
    merged_df['tackles_new'].notna(), 
    merged_df['tackles_new'], 
    np.where(merged_df['minutes'] == 0, 0, merged_df['tackles'].fillna(0))
)

# Update CBI
old_cbi = merged_df['clearances_blocks_interceptions'] if 'clearances_blocks_interceptions' in merged_df.columns else 0
merged_df['clearances_blocks_interceptions'] = np.where(
    merged_df['cbi_new'].notna(), 
    merged_df['cbi_new'], 
    np.where(merged_df['minutes'] == 0, 0, old_cbi)
)

# Clean up temps - keep game_number as it's useful for feature engineering
merged_df.drop(columns=['tackles_new', 'cbi_new', 'join_name', 'join_season', 'is_matched', 'key_tuple'], inplace=True, errors='ignore')

print("\n✓ Merge complete using game_number for correct chronological alignment")

In [ ]:
# ==========================================
# STEP 4: COMBINE MERGED DATA BACK INTO FULL DATASET
# ==========================================
# Problem: We filtered df_main to only seasons with defensive data (2019-20 to 2024-25)
# Solution: Reload the full dataset and combine:
#   - Seasons 2016-17 to 2018-19: Already have real defensive stats
#   - Seasons 2019-20 to 2024-25: Use merged_df with newly merged defensive stats
#   - Season 2025-26: Keep as-is (if not in defensive data)

print("="*80)
print("STEP 4: Combining merged data back into full dataset")
print("="*80)

# 1. Reload the full all_seasons_data (before we filtered it)
df_full = pd.read_csv("all_seasons_data.csv")
print(f"Full dataset shape: {df_full.shape}")
print(f"Seasons in full dataset: {sorted(df_full['season'].unique())}")

# 2. Get the seasons that were merged with defensive data
merged_seasons = merged_df['season'].unique().tolist()
print(f"\nSeasons that were merged with defensive stats: {merged_seasons}")

# 3. Get the seasons that were NOT merged (already have defensive data or no data available)
seasons_not_merged = [s for s in df_full['season'].unique() if s not in merged_seasons]
print(f"Seasons NOT merged (already have defensive data): {seasons_not_merged}")

# 4. Extract the non-merged seasons from the full dataset
df_not_merged = df_full[df_full['season'].isin(seasons_not_merged)].copy()
print(f"Records from non-merged seasons: {len(df_not_merged):,}")

# 5. Ensure both DataFrames have the same columns
# Add game_number to non-merged seasons if missing (set to None for older seasons)
if 'game_number' not in df_not_merged.columns:
    df_not_merged['game_number'] = None
    
# Align columns - get the union of both column sets
all_columns = list(set(merged_df.columns) | set(df_not_merged.columns))
for col in all_columns:
    if col not in merged_df.columns:
        merged_df[col] = None
    if col not in df_not_merged.columns:
        df_not_merged[col] = None

# 6. Combine merged seasons with non-merged seasons
all_seasons_final = pd.concat([df_not_merged, merged_df], ignore_index=True)

# 7. Sort by season and element for consistency
all_seasons_final = all_seasons_final.sort_values(['season', 'element', 'GW']).reset_index(drop=True)

print(f"\n✅ Final combined dataset shape: {all_seasons_final.shape}")
print(f"Seasons in final dataset: {sorted(all_seasons_final['season'].unique())}")

# 8. Verify record counts by season
print("\nRecords per season:")
print(all_seasons_final.groupby('season').size().sort_index())

In [ ]:
# ==========================================
# STEP 5: RECALCULATE DEFENSIVE CONTRIBUTION & MODIFY POINTS
# ==========================================
# Now that we have the real defensive stats for 2019-20 to 2024-25,
# we need to recalculate defensive_contribution and apply the point modifications

print("="*80)
print("STEP 5: Recalculating defensive contribution for merged seasons")
print("="*80)

# 1. Define the defensive contribution calculation function
def calculate_defensive_contribution_row(row):
    """Calculate defensive contribution based on position"""
    position = row['position']
    cbi = row.get('clearances_blocks_interceptions', 0) or 0
    tackles = row.get('tackles', 0) or 0
    recoveries = row.get('recoveries', 0) or 0
    
    if position == 'DEF':
        return cbi + tackles
    elif position in ['MID', 'FWD']:
        return cbi + tackles + recoveries
    else:  # GK or unknown
        return 0

# 2. Recalculate defensive_contribution for seasons that were merged
# (2019-20 to 2024-25 now have real defensive stats)
merged_season_mask = all_seasons_final['season'].isin(merged_seasons)

print(f"Recalculating defensive_contribution for {merged_season_mask.sum():,} records...")

all_seasons_final.loc[merged_season_mask, 'defensive_contribution'] = \
    all_seasons_final.loc[merged_season_mask].apply(calculate_defensive_contribution_row, axis=1)

# 3. Apply point modification to merged seasons (as per FPL rules)
# Add 2 bonus points when:
# - Defenders reach defensive_contribution >= 10
# - Midfielders/Forwards reach defensive_contribution >= 12

def calculate_modified_points(row):
    """Add 2 bonus points for defensive contribution threshold"""
    points = row['total_points']
    position = row['position']
    def_contrib = row.get('defensive_contribution', 0) or 0
    
    if position == 'DEF' and def_contrib >= 10:
        points += 2
    elif position in ['MID', 'FWD'] and def_contrib >= 12:
        points += 2
    return points

# Only modify points for the merged seasons (2019-20 to 2024-25)
# Seasons 2016-17 to 2018-19 already had points modified earlier
print("Applying point modifications for defensive contributions...")

all_seasons_final.loc[merged_season_mask, 'total_points'] = \
    all_seasons_final.loc[merged_season_mask].apply(calculate_modified_points, axis=1)

print("✅ Defensive contribution recalculated and points modified")

# 4. Verify the results
print("\nSample of defensive stats after recalculation:")
sample_cols = ['name', 'season', 'position', 'tackles', 'clearances_blocks_interceptions', 
               'defensive_contribution', 'total_points']
available_cols = [c for c in sample_cols if c in all_seasons_final.columns]
print(all_seasons_final[all_seasons_final['season'] == '2023-24'][available_cols].head(10))

In [ ]:
# ==========================================
# STEP 6: SAVE THE FINAL COMPLETE DATASET
# ==========================================
print("="*80)
print("STEP 6: Saving final dataset")
print("="*80)

# Save the complete dataset with all seasons and merged defensive stats
output_path = 'all_seasons_data_final.csv'
all_seasons_final.to_csv(output_path, index=False, encoding='utf-8')

print(f"✅ Saved: {output_path}")
print(f"   Total records: {len(all_seasons_final):,}")
print(f"   Total columns: {len(all_seasons_final.columns)}")
print(f"   Seasons: {sorted(all_seasons_final['season'].unique())}")

# Summary statistics
print("\n" + "="*80)
print("FINAL DATASET SUMMARY")
print("="*80)
print(f"\nRecords by season:")
print(all_seasons_final.groupby('season').size().sort_index())

print(f"\nDefensive stats coverage (non-zero defensive_contribution):")
def_contrib_stats = all_seasons_final.groupby('season').apply(
    lambda x: (x['defensive_contribution'] > 0).sum() / len(x) * 100
)
print(def_contrib_stats.round(1).to_string())

print("\n" + "="*80)
print("DATA PREPARATION COMPLETE - READY FOR FEATURE ENGINEERING")
print("="*80)